# Trabalho 1, Tema 3: Anticiclone Subtropical do Atlântico Sul (ASAS) → fluxos ar-mar → TSM, salinidade e camada de mistura

**ACA0430 — Meteorologia Sinótica e Aplicações à Oceanografia (IAG/USP), 2º semestre de 2026**

**Objetivo.** Montar as figuras de base de um estudo de caso em que uma **anomalia do ASAS**, de escala mensal ou sazonal, altera os **fluxos de calor e de água doce** entre a atmosfera e o oceano e, com isso, a **TSM**, a **salinidade** e a **profundidade da camada de mistura**. O exemplo padrão é o **verão (janeiro–fevereiro) de 2014**: um ASAS anômalo e persistente ("bloqueio") associado à seca no Sudeste do Brasil e a uma **onda de calor marinha** no Atlântico Sul (Rodrigues et al., 2019).

**Desenho.** Aqui o "caso" não é um evento de poucos dias, mas uma **anomalia mensal**: comparamos a média do período do caso com uma **climatologia** do mesmo período do ano.

**Ligação com as aulas.** Sob o ASAS:

- **subsidência** → céu claro → **mais radiação de onda curta** chega à superfície;
- ar seco e vento → **mais perda de calor latente** e **mais evaporação** → **aumento da salinidade** (E − P > 0);
- **convergência de Ekman** sob o anticiclone → aprofundamento ou estabilização da camada de mistura.

**Mas o efeito pode ser outro.** No verão de 2014, uma anomalia anticiclônica persistente (bloqueio atmosférico) sobre o Atlântico Sul ocidental esteve associada à seca no Sudeste do Brasil (Coelho et al., 2016; Rodrigues & Woollings, 2017). Rodrigues et al. (2019) relacionam essa mesma anomalia à onda de calor marinha no Atlântico Sul, com mais radiação de onda curta (menos nuvens) e **vento mais fraco**, que reduz a perda de calor latente do oceano. Silva & Dottori (2021) discutem a influência desse bloqueio sobre a plataforma continental do Sudeste. O ASAS em si é descrito em Sun et al. (2017) e Reboita et al. (2019).

**A resposta não precisa ser igual em toda a região.** Ao analisar as figuras, compare o centro da bacia com a faixa costeira do Sudeste: o vento enfraqueceu nos dois lugares? Qual termo do fluxo de calor domina em cada um? A camada de mistura ficou mais rasa ou mais funda? Lembre que ela depende do vento (mistura mecânica), do fluxo de calor e também da salinidade, porque água de superfície mais salina é mais densa e menos estável.

**Este notebook é um ponto de partida**, não uma receita fechada.

**Dados:** ERA5 (atmosfera, 0,25°, horário, subamostrado) e GLORYS12 (oceano, médias mensais e climatologia mensal, 1/12°), lidos de armazenamentos **Zarr na nuvem** (ARCO).

## 1. Instalação dos pacotes

O Colab já traz `numpy`, `pandas`, `xarray` e `matplotlib`. Os pacotes abaixo **não vêm pré-instalados**:

- `copernicusmarine`: *toolbox* oficial do Copernicus Marine (GLORYS12 e WAVERYS);
- `gcsfs` e `zarr`: leitura do ERA5 direto do Google Cloud Storage, em formato Zarr;
- `cartopy`: mapas com linha de costa e projeções;
- `cmocean`: paletas de cores pensadas para oceanografia.

A instalação leva cerca de 1 minuto. Em geral **não é preciso reiniciar** a sessão. Se aparecer erro de importação na célula seguinte, use *Ambiente de execução → Reiniciar sessão* e rode o notebook de novo a partir daqui.

Fora do Colab (no seu computador), a instalação automática é pulada: instale os pacotes no seu ambiente antes.

In [ ]:
import sys

# Detecta se estamos no Google Colab
EM_COLAB = "google.colab" in sys.modules

# Pacotes que não vêm pré-instalados no Colab
if EM_COLAB:
    !pip install -q copernicusmarine gcsfs zarr cartopy cmocean

## 2. Importações

Importamos tudo o que vamos usar e imprimimos as versões dos pacotes principais. **Se algo der errado, copie esta saída ao pedir ajuda**: muitos problemas vêm de diferenças de versão.

In [ ]:
import os
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean
import zarr
import gcsfs
import copernicusmarine

warnings.filterwarnings("ignore", category=FutureWarning)

for nome, modulo in [("xarray", xr), ("zarr", zarr), ("gcsfs", gcsfs),
                     ("copernicusmarine", copernicusmarine), ("cartopy", cartopy)]:
    print(f"{nome:18s} {modulo.__version__}")

## 3. Credenciais do Copernicus Marine (CMEMS)

O **ERA5** que usamos aqui é **público**: não precisa de conta. Já o **CMEMS** (GLORYS12 e WAVERYS) exige uma **conta gratuita**.

**Nunca escreva sua senha no notebook.** Se você compartilhar o notebook, a senha vai junto.

### Passo a passo (uma vez só)

1. **Crie a conta** em https://data.marine.copernicus.eu/register. Preencha nome, e-mail, usuário e senha e confirme pelo link que chega no e-mail. Guarde o **nome de usuário**, que não é o e-mail.
2. **Crie os segredos no Colab:**
   - na barra lateral esquerda do Colab, clique no ícone de **chave** ("Secrets" / "Segredos");
   - clique em **"+ Adicionar novo segredo"**;
   - em *Nome*, escreva `COPERNICUSMARINE_SERVICE_USERNAME` e, em *Valor*, seu usuário do Copernicus Marine;
   - adicione um segundo segredo, `COPERNICUSMARINE_SERVICE_PASSWORD`, com a sua senha;
   - ative a chave **"Acesso ao notebook"** nos dois segredos.
3. Rode a célula abaixo. Se o Colab perguntar se o notebook pode acessar os segredos, clique em **"Conceder acesso"**.

Se você não criar os segredos, a célula pede usuário e senha numa caixa de texto (`getpass`), que não fica gravada no notebook.

Os segredos ficam na sua conta Google e servem para todos os seus notebooks do Colab.

In [ ]:
usuario = os.environ.get("COPERNICUSMARINE_SERVICE_USERNAME")
senha = os.environ.get("COPERNICUSMARINE_SERVICE_PASSWORD")

# 1) Segredos do Colab (ícone de chave na barra lateral)
if EM_COLAB and not usuario:
    from google.colab import userdata
    try:
        usuario = userdata.get("COPERNICUSMARINE_SERVICE_USERNAME")
        senha = userdata.get("COPERNICUSMARINE_SERVICE_PASSWORD")
    except Exception:  # segredo não criado ou acesso não concedido
        usuario, senha = None, None

if not usuario and copernicusmarine.login(check_credentials_valid=True):
    # Fora do Colab: login já salvo neste computador (comando `copernicusmarine login`)
    print("Usando o login do Copernicus Marine já salvo neste computador.")
else:
    # 2) Alternativa: perguntar usuário e senha (a senha não aparece na tela)
    if not usuario:
        from getpass import getpass
        usuario = input("Usuário do Copernicus Marine: ")
        senha = getpass("Senha do Copernicus Marine: ")

    # O toolbox também lê estas variáveis de ambiente
    os.environ["COPERNICUSMARINE_SERVICE_USERNAME"] = usuario
    os.environ["COPERNICUSMARINE_SERVICE_PASSWORD"] = senha

    # 3) Login (uma vez por sessão)
    ok = copernicusmarine.login(username=usuario, password=senha, force_overwrite=True)
    print("Login OK" if ok else "Falha no login: confira usuário e senha")

del senha  # não deixa a senha numa variável visível

## 4. Parâmetros do estudo de caso

**Esta é a única célula que você precisa alterar.**

- `REGIAO`: Atlântico Sul ocidental (45°S–10°S, 60°W–10°W).
- `ANO`, `MESES`: o período do caso (padrão: janeiro e fevereiro de 2014).
- `HORAS_UTC`: horários usados do ERA5 em cada dia do caso. O armazenamento ARCO é **horário** e não tem médias mensais. Para reduzir o volume, **subamostramos**: 4 horários por dia em vez de 24.
- `ANOS_CLIM`: anos usados na "climatologia" do ERA5 (**no máximo \~10 anos**). Nela usamos **1 instante por dia**, alternando entre os `HORAS_UTC` de um dia para o outro. Assim caso e climatologia amostram o **mesmo ciclo diurno**, o que importa muito para a radiação solar.
- `CAIXA`: área para a série mensal opcional (seção 10.5).

**A climatologia do ERA5 é aproximada** (poucos anos, subamostrada). A do GLORYS12 é a climatologia mensal oficial do produto (1993–2016). Os dois períodos de referência são diferentes: diga isso nos Métodos.

In [ ]:
# >>> ALUNO: domínio
REGIAO = dict(lon_o=-60.0, lon_l=-10.0, lat_s=-45.0, lat_n=-10.0)

# >>> ALUNO: período do caso
ANO = 2014
MESES = [1, 2]

# >>> ALUNO: horários (UTC) do ERA5 usados em cada dia
HORAS_UTC = [0, 6, 12, 18]

# >>> ALUNO: anos da climatologia aproximada do ERA5 (no máximo ~10)
ANOS_CLIM = list(range(2004, 2014))

# >>> ALUNO (opcional): caixa para a série mensal da seção 10.5
CAIXA = dict(lon_o=-50.0, lon_l=-35.0, lat_s=-30.0, lat_n=-20.0)
FAZER_SERIE_CAIXA = True

NOMES_MESES = {1: "jan", 2: "fev", 3: "mar", 4: "abr", 5: "mai", 6: "jun",
               7: "jul", 8: "ago", 9: "set", 10: "out", 11: "nov", 12: "dez"}
TEXTO_CASO = "–".join(NOMES_MESES[m] for m in MESES) + f"/{ANO}"
TEXTO_CLIM = f"{ANOS_CLIM[0]}–{ANOS_CLIM[-1]}"

def dias_dos_meses(ano, meses):
    """Lista de dias (00 UTC) dos meses pedidos em um ano."""
    dias = pd.date_range(f"{ano}-01-01", f"{ano}-12-31", freq="D")
    return dias[dias.month.isin(meses)]

# Instantes do caso: todos os HORAS_UTC de cada dia
DATAS_CASO = pd.DatetimeIndex([d + pd.Timedelta(hours=h) for d in dias_dos_meses(ANO, MESES)
                               for h in HORAS_UTC])

# Instantes da climatologia: 1 por dia, alternando os horários entre os dias
DATAS_CLIM = {}
for a in ANOS_CLIM:
    dias = dias_dos_meses(a, MESES)
    DATAS_CLIM[a] = pd.DatetimeIndex([d + pd.Timedelta(hours=HORAS_UTC[i % len(HORAS_UTC)])
                                      for i, d in enumerate(dias)])

print(f"Caso: {TEXTO_CASO}, {len(DATAS_CASO)} instantes do ERA5")
print(f"Climatologia ERA5: {TEXTO_CLIM}, {sum(len(v) for v in DATAS_CLIM.values())} instantes")

### Pastas de saída

No Colab, os recortes de dados (NetCDF) vão para `/content/dados/` e as figuras para `/content/figuras/`. **Atenção: a pasta `/content` é apagada quando a sessão termina.** No fim do notebook há uma célula opcional para copiar tudo para o seu Google Drive.

In [ ]:
PASTA_BASE = Path("/content") if EM_COLAB else Path.cwd() / "saida"
PASTA_DADOS = PASTA_BASE / "dados"
PASTA_FIGURAS = PASTA_BASE / "figuras"
PASTA_DADOS.mkdir(parents=True, exist_ok=True)
PASTA_FIGURAS.mkdir(parents=True, exist_ok=True)
print("Dados:  ", PASTA_DADOS)
print("Figuras:", PASTA_FIGURAS)

## 5. Funções auxiliares

Poucas funções, usadas várias vezes ao longo do notebook:

- `lon_para_360` / `lon_para_180`: o ERA5 usa longitude de **0 a 360°** (o Brasil fica entre \~305° e \~330°). Recortamos com a longitude convertida e só voltamos para −180..180 **depois** do recorte, na hora de plotar;
- `checar_tamanho`: mostra quantos MB o recorte vai ocupar **antes** de carregar;
- `mapa_base`: costa, fronteiras, grade e (opcionalmente) continente em cinza;
- `passo_vetores`: reduz o número de setas para o mapa não virar um borrão;
- `ponto_oceano_mais_proximo`: perto da costa, a grade dos modelos oceânicos tem pontos de terra (NaN). Esta função escolhe o ponto **de oceano** mais próximo;
- `rodape` e `salvar_figura`: fonte dos dados e exportação em PNG.

In [ ]:
def lon_para_360(lon):
    """Converte longitude de -180..180 para 0..360 (convenção do ERA5)."""
    return lon % 360


def lon_para_180(ds, nome="longitude"):
    """Converte a coordenada de longitude de 0..360 para -180..180 e reordena (para plotar)."""
    ds = ds.assign_coords({nome: ((ds[nome] + 180) % 360) - 180})
    return ds.sortby(nome)


def checar_tamanho(ds, nome, limite_mb=1500):
    """Imprime o tamanho estimado do recorte ANTES de carregar na memória."""
    mb = ds.nbytes / 1e6
    print(f"{nome}: {mb:.1f} MB")
    if mb > limite_mb:
        print(f"  ATENÇÃO: mais de {limite_mb} MB! Reduza o período ou a área antes de carregar.")
    return mb


def mapa_base(ax, regiao, terra=True):
    """Prepara um eixo cartopy: extensão, costa, grade com rótulos.

    terra=True pinta os continentes de cinza POR CIMA dos campos (para campos oceânicos).
    """
    ax.set_extent([regiao["lon_o"], regiao["lon_l"], regiao["lat_s"], regiao["lat_n"]],
                  crs=ccrs.PlateCarree())
    if terra:
        ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="0.85", zorder=3)
    ax.coastlines("50m", linewidth=0.6, zorder=4)
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.3, edgecolor="0.4", zorder=4)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="0.5", linestyle="--", zorder=5)
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 7}
    gl.ylabel_style = {"size": 7}
    return ax


def passo_vetores(n_pontos, n_setas=20):
    """Quantos pontos pular para desenhar ~n_setas vetores por eixo."""
    return max(1, n_pontos // n_setas)


def barra_cores(fig, mapeavel, rotulo):
    """Ajusta o espaço entre os painéis e põe UMA barra de cores horizontal embaixo da figura.

    Devolve a altura (fração da figura) da barra, útil para posicionar a seta de referência.
    """
    h = fig.get_figheight()
    y_barra = 0.8 / h
    fig.subplots_adjust(left=0.04, right=0.98, top=1 - 0.4 / h, bottom=y_barra + 1.0 / h,
                        wspace=0.18, hspace=0.3)
    cax = fig.add_axes([0.25, y_barra, 0.5, 0.15 / h])
    fig.colorbar(mapeavel, cax=cax, orientation="horizontal", label=rotulo)
    return y_barra


def rodape(fig, texto):
    """Escreve a fonte dos dados no rodapé da figura."""
    fig.text(0.01, 0.005, texto, fontsize=7, color="0.3", ha="left", va="bottom")


def ponto_oceano_mais_proximo(campo, lon, lat, nome_lon="longitude", nome_lat="latitude"):
    """Ponto de grade mais próximo de (lon, lat) que NÃO é terra (valor não-NaN).

    `campo` é um DataArray 2D (lat, lon) em que a terra está como NaN.
    Devolve (lon, lat) do ponto escolhido.
    """
    coslat = np.cos(np.deg2rad(lat))
    dist2 = ((campo[nome_lon] - lon) * coslat) ** 2 + (campo[nome_lat] - lat) ** 2
    dist2 = dist2.where(campo.notnull())
    idx = dist2.argmin(dim=[nome_lat, nome_lon])
    lon_ok = float(campo[nome_lon][idx[nome_lon]])
    lat_ok = float(campo[nome_lat][idx[nome_lat]])
    return lon_ok, lat_ok


def salvar_figura(fig, nome):
    """Salva a figura em PNG (200 dpi) na pasta de figuras."""
    caminho = PASTA_FIGURAS / nome
    fig.savefig(caminho, dpi=200, bbox_inches="tight")
    print("Figura salva:", caminho)

## 6. ERA5: abertura do armazenamento ARCO (Zarr na nuvem)

O **ARCO-ERA5** é uma cópia do ERA5 no Google Cloud em formato **Zarr**, *Analysis-Ready, Cloud-Optimized* (Abernathey et al., 2021). Em vez de baixar arquivos inteiros, abrimos o conjunto completo (1940 até hoje, horário, 0,25°) de forma **preguiçosa**: só os metadados são lidos agora. Os números só são transferidos quando chamamos `.load()`.

O armazenamento é dividido em pedaços (*chunks*) de **um instante × globo inteiro**, ótimo para **mapas**. Por isso:

- abrimos o armazenamento **uma única vez** (leva alguns segundos);
- recortamos **no espaço e no tempo antes** de carregar;
- nunca fazemos um laço hora a hora reabrindo o armazenamento.

O ERA5 "final" tem cerca de 3 meses de atraso. Os meses mais recentes são o **ERA5T** (preliminar), que pode ser corrigido depois.

In [ ]:
URL_ERA5 = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"

era5 = xr.open_zarr(URL_ERA5, chunks=None, storage_options=dict(token="anon"))

print("ERA5 final:          ", era5.attrs["valid_time_start"], "a", era5.attrs["valid_time_stop"])
print("ERA5T (preliminar) até", era5.attrs["valid_time_stop_era5t"])
print("Número de variáveis: ", len(era5.data_vars))

### Nomes das variáveis

As variáveis do ARCO-ERA5 têm **nomes longos** (`mean_sea_level_pressure`, `10m_u_component_of_wind`...). Os nomes curtos do ECMWF (`msl`, `u10`) só aparecem nos atributos. A célula abaixo lista as variáveis com seus nomes curtos e unidades. Use-a para procurar outras variáveis que queira acrescentar.

In [ ]:
# >>> ALUNO: troque as palavras-chave para procurar outras variáveis
PALAVRAS = ["sea_level", "10m_", "sea_surface", "heat_flux", "radiation", "evaporation",
            "precipitation", "stress", "geopotential", "land_sea"]

for v in sorted(era5.data_vars):
    if any(p in v for p in PALAVRAS):
        atr = era5[v].attrs
        print(f"{v:58s} {atr.get('short_name', '?'):10s} [{atr.get('units', '?')}]")

### 6.1 Variáveis e convenções dos fluxos

| Variável no ARCO-ERA5 | Nome curto | Unidade no armazenamento | Significado |
|---|---|---|---|
| `mean_sea_level_pressure` | msl | Pa | pressão ao nível do mar |
| `10m_u_component_of_wind`, `10m_v_component_of_wind` | u10, v10 | m s⁻¹ | vento a 10 m |
| `surface_net_solar_radiation` | ssr | J m⁻² | onda curta líquida |
| `surface_net_thermal_radiation` | str | J m⁻² | onda longa líquida |
| `surface_sensible_heat_flux` | sshf | J m⁻² | calor sensível |
| `surface_latent_heat_flux` | slhf | J m⁻² | calor latente |
| `evaporation` | e | m (de água) | evaporação |
| `total_precipitation` | tp | m (de água) | precipitação total |

**Fluxos e precipitação são acumulados.** No ERA5 horário, cada valor é o total acumulado **na hora que termina** no instante indicado. Por isso:

- fluxo em W m⁻² = valor em J m⁻² ÷ 3600 s;
- taxa em mm/dia = valor em m × 1000 mm/m × 24 h/dia.

**Convenção de sinal do ECMWF: positivo para baixo**, isto é, fluxo **para o oceano**. Assim:

- onda curta (ssr) é sempre ≥ 0 (o oceano ganha);
- onda longa (str), sensível (sshf) e latente (slhf) costumam ser **negativos** (o oceano perde calor);
- **fluxo líquido** Q = ssr + str + sshf + slhf (Q > 0: o oceano ganha calor);
- a **evaporação** também segue "positivo para baixo": `e` é **negativo** quando há evaporação. Para ter E positivo, usamos E = −e. Então **E − P = −e − tp**.

Mantemos essa convenção nos rótulos de todas as figuras.

In [ ]:
VARS_ERA5 = {
    "mean_sea_level_pressure": "msl",
    "10m_u_component_of_wind": "u10",
    "10m_v_component_of_wind": "v10",
    "surface_net_solar_radiation": "ssr",
    "surface_net_thermal_radiation": "str",
    "surface_sensible_heat_flux": "sshf",
    "surface_latent_heat_flux": "slhf",
    "evaporation": "e",
    "total_precipitation": "tp",
}
for nome in VARS_ERA5:
    print(f"{nome:32s} [{era5[nome].attrs.get('units')}]")


def recorte_era5(datas):
    """Recorte (preguiçoso) do ERA5 na REGIAO para uma lista de instantes."""
    return era5[list(VARS_ERA5)].sel(
        time=datas,
        latitude=slice(REGIAO["lat_n"], REGIAO["lat_s"]),
        longitude=slice(lon_para_360(REGIAO["lon_o"]), lon_para_360(REGIAO["lon_l"])),
    ).rename(VARS_ERA5)


def converte_unidades(ds):
    """Pa → hPa; J m-2 (acumulado em 1 h) → W m-2; m (acumulado em 1 h) → mm/dia."""
    ds = ds.copy()
    ds["msl"] = ds["msl"] / 100.0
    for v in ["ssr", "str", "sshf", "slhf"]:
        ds[v] = ds[v] / 3600.0
    ds["Q"] = ds["ssr"] + ds["str"] + ds["sshf"] + ds["slhf"]
    ds["EmP"] = (-ds["e"] - ds["tp"]) * 1000.0 * 24.0      # E - P (mm/dia)
    return ds.drop_vars(["e", "tp"])


# Máscara terra-mar (um instante basta)
lsm = era5["land_sea_mask"].sel(
    time=DATAS_CASO[0],
    latitude=slice(REGIAO["lat_n"], REGIAO["lat_s"]),
    longitude=slice(lon_para_360(REGIAO["lon_o"]), lon_para_360(REGIAO["lon_l"])),
).drop_vars("time").load()
lsm = lon_para_180(lsm)
oceano = lsm < 0.5

### 6.2 Média do caso

Carregamos os instantes do caso e fazemos a média no tempo. Antes, imprimimos o volume estimado.

In [ ]:
e5_caso = recorte_era5(DATAS_CASO)
checar_tamanho(e5_caso, f"ERA5 caso ({TEXTO_CASO})")
e5_caso = converte_unidades(e5_caso.load())
media_caso = lon_para_180(e5_caso.mean("time"))
media_caso.to_netcdf(PASTA_DADOS / f"t3_era5_media_caso_{ANO}.nc")

### 6.3 Climatologia aproximada

O volume total é impresso **antes** de carregar. Para não ocupar muita memória, carregamos **um ano por vez**, guardamos só a média desse ano e descartamos o resto. Isso não é um laço hora a hora: cada ano é **um único** pedido ao armazenamento já aberto.

Esta é a célula mais demorada do notebook (alguns minutos por ano no Colab). Para testar, reduza `ANOS_CLIM`.

In [ ]:
mb_total = sum(recorte_era5(d).nbytes for d in DATAS_CLIM.values()) / 1e6
print(f"Volume total da climatologia: {mb_total:.0f} MB (carregado um ano por vez)")

medias_anuais = []
for a, datas in DATAS_CLIM.items():
    sub = converte_unidades(recorte_era5(datas).load())
    medias_anuais.append(sub.mean("time"))
    print(f"  {a}: {len(datas)} instantes ok")
clim = lon_para_180(xr.concat(medias_anuais, dim="ano").mean("ano"))
clim.to_netcdf(PASTA_DADOS / f"t3_era5_clim_{ANOS_CLIM[0]}_{ANOS_CLIM[-1]}.nc")

anom = media_caso - clim
anom["vel10"] = np.hypot(media_caso["u10"], media_caso["v10"]) - np.hypot(clim["u10"], clim["v10"])

## 7. CMEMS: GLORYS12 mensal e climatologia mensal

- **Médias mensais:** `cmems_mod_glo_phy_my_0.083deg_P1M-m`.
- **Climatologia mensal (1993–2016):** `cmems_mod_glo_phy_my_0.083deg-climatology_P1M-m`.

Variáveis: `thetao` e `so` no nível mais raso (\~0,5 m), que usamos como TSM e salinidade de superfície (SSS), e `mlotst`, a **espessura da camada de mistura** definida por um critério de densidade (sigma-theta), em m.

Usamos o serviço `arco-geo-series` (mapas). O aviso `WARNING ... depth` é esperado.

In [ ]:
def abre_glorys(dataset_id, inicio, fim, regiao):
    ds = copernicusmarine.open_dataset(
        dataset_id=dataset_id,
        variables=["thetao", "so", "mlotst"],
        minimum_longitude=regiao["lon_o"], maximum_longitude=regiao["lon_l"],
        minimum_latitude=regiao["lat_s"], maximum_latitude=regiao["lat_n"],
        minimum_depth=0, maximum_depth=1,
        start_datetime=inicio, end_datetime=fim,
        service="arco-geo-series",
        coordinates_selection_method="outside",
    )
    # "outside" pode trazer um mês a mais nas pontas (ex.: jan do ano seguinte): cortamos
    return ds.isel(depth=0).sel(time=slice(inicio, fim))

ID_MENSAL = "cmems_mod_glo_phy_my_0.083deg_P1M-m"
ID_CLIM = "cmems_mod_glo_phy_my_0.083deg-climatology_P1M-m"

g_mensal = abre_glorys(ID_MENSAL, f"{ANO}-01-01", f"{ANO}-12-31", REGIAO)
g_mensal = g_mensal.sel(time=g_mensal.time.dt.month.isin(MESES))
checar_tamanho(g_mensal, "GLORYS12 mensal (caso)")
g_mensal = g_mensal.load()

g_clim = copernicusmarine.open_dataset(
    dataset_id=ID_CLIM, variables=["thetao", "so", "mlotst"],
    minimum_longitude=REGIAO["lon_o"], maximum_longitude=REGIAO["lon_l"],
    minimum_latitude=REGIAO["lat_s"], maximum_latitude=REGIAO["lat_n"],
    minimum_depth=0, maximum_depth=1,
    service="arco-geo-series", coordinates_selection_method="outside",
).isel(depth=0)
print("Instantes da climatologia:", g_clim.time.values[:3], "...")
g_clim = g_clim.sel(time=g_clim.time.dt.month.isin(MESES))
checar_tamanho(g_clim, "GLORYS12 climatologia (meses do caso)")
g_clim = g_clim.load()

g_anom = g_mensal.mean("time") - g_clim.mean("time")
g_mensal.to_netcdf(PASTA_DADOS / f"t3_glorys_mensal_{ANO}.nc")
g_clim.to_netcdf(PASTA_DADOS / "t3_glorys_climatologia.nc")
for v in g_mensal.data_vars:
    print(f"{v:7s} [{g_mensal[v].attrs.get('units')}]  {g_mensal[v].attrs.get('standard_name')}")

## 8. Situação atmosférica: PNM e vento

Três painéis:

1. **Média do caso:** PNM (isolinhas) e vento a 10 m (vetores).
2. **Anomalia de PNM** (cores) com as **isóbaras da climatologia** (isolinhas cinza).
3. **Anomalia do vento:** anomalia da intensidade (cores) e vetor anomalia (setas).

**O que observar:** o ASAS está mais intenso? Deslocado para oeste/sul? O vento sobre o oceano junto à costa SE ficou mais fraco ou mais forte?

In [ ]:
n = passo_vetores(media_caso.sizes["longitude"])
lon, lat = media_caso.longitude, media_caso.latitude
niveis_pnm = np.arange(1000, 1036, 2)

fig, axs = plt.subplots(1, 3, figsize=(17, 5), subplot_kw=dict(projection=ccrs.PlateCarree()))

ax = axs[0]
mapa_base(ax, REGIAO, terra=False)
cf = ax.contourf(lon, lat, np.hypot(media_caso["u10"], media_caso["v10"]), levels=np.arange(0, 11, 1),
                 cmap=cmocean.cm.speed, extend="max", transform=ccrs.PlateCarree())
cs = ax.contour(lon, lat, media_caso["msl"], levels=niveis_pnm, colors="k", linewidths=0.8,
                transform=ccrs.PlateCarree())
ax.clabel(cs, fmt="%d", fontsize=7)
q = ax.quiver(lon[::n], lat[::n], media_caso["u10"][::n, ::n], media_caso["v10"][::n, ::n],
              scale=150, width=0.003, transform=ccrs.PlateCarree())
ax.quiverkey(q, 0.9, 1.04, 5, "5 m s$^{-1}$", labelpos="E", fontproperties={"size": 7})
fig.colorbar(cf, ax=ax, orientation="horizontal", pad=0.08, shrink=0.85,
             label="Intensidade do vento médio a 10 m (m s$^{-1}$)")
ax.set_title(f"PNM (hPa) e vento 10 m — média {TEXTO_CASO}", fontsize=9, loc="left")

ax = axs[1]
mapa_base(ax, REGIAO, terra=False)
lim = np.ceil(float(np.abs(anom["msl"]).max()))
cf = ax.contourf(lon, lat, anom["msl"], levels=np.linspace(-lim, lim, 21), cmap=cmocean.cm.balance,
                 transform=ccrs.PlateCarree())
cs = ax.contour(lon, lat, clim["msl"], levels=niveis_pnm, colors="0.35", linewidths=0.8,
                linestyles="--", transform=ccrs.PlateCarree())
ax.clabel(cs, fmt="%d", fontsize=7)
fig.colorbar(cf, ax=ax, orientation="horizontal", pad=0.08, shrink=0.85,
             label=f"Anomalia de PNM (hPa) vs. {TEXTO_CLIM}")
ax.set_title(f"Anomalia de PNM e isóbaras climatológicas ({TEXTO_CLIM})", fontsize=9)

ax = axs[2]
mapa_base(ax, REGIAO, terra=False)
lim = np.ceil(float(np.abs(anom["vel10"]).max()))
cf = ax.contourf(lon, lat, anom["vel10"], levels=np.linspace(-lim, lim, 21), cmap=cmocean.cm.balance,
                 transform=ccrs.PlateCarree())
q = ax.quiver(lon[::n], lat[::n], anom["u10"][::n, ::n], anom["v10"][::n, ::n],
              scale=60, width=0.003, transform=ccrs.PlateCarree())
ax.quiverkey(q, 0.9, 1.04, 2, "2 m s$^{-1}$", labelpos="E", fontproperties={"size": 7})
fig.colorbar(cf, ax=ax, orientation="horizontal", pad=0.08, shrink=0.85,
             label="Anomalia da intensidade do vento a 10 m (m s$^{-1}$)")
ax.set_title("Anomalia do vento a 10 m", fontsize=9, loc="left")

rodape(fig, f"Fonte: ERA5 (Hersbach et al., 2020), via ARCO-ERA5. Caso: {TEXTO_CASO}, {len(HORAS_UTC)} horários/dia; "
            f"climatologia aproximada {TEXTO_CLIM}, 1 instante/dia.")
salvar_figura(fig, f"t3_pnm_vento_anomalia_{ANO}.png")
plt.show()

## 9. Anomalias dos fluxos de calor (ERA5)

Painéis com as anomalias (caso − climatologia) de onda curta, onda longa, calor sensível e calor latente, e o **fluxo líquido** Q. Todos em W m⁻², **positivo para baixo** (anomalia positiva = o oceano ganha mais calor, ou perde menos, que o normal), e com a mesma escala de cores. O continente está mascarado.

**O que observar:** onde a anomalia de Q é positiva? Ela vem principalmente da **onda curta** (menos nuvens) ou do **calor latente** (vento mais fraco, menos evaporação)? Compare com o mapa de anomalia do vento.

In [ ]:
paineis = [("ssr", "Onda curta líquida"), ("str", "Onda longa líquida"),
           ("sshf", "Calor sensível"), ("slhf", "Calor latente"), ("Q", "Fluxo líquido Q")]
lim = float(max(np.nanpercentile(np.abs(anom[v].where(oceano)), 99) for v, _ in paineis))
lim = np.ceil(lim / 10) * 10
niveis_q = np.linspace(-lim, lim, 21)

fig, axs = plt.subplots(2, 3, figsize=(15, 9), subplot_kw=dict(projection=ccrs.PlateCarree()))
for ax, (v, titulo) in zip(axs.flat, paineis):
    mapa_base(ax, REGIAO, terra=True)
    cf = ax.contourf(anom.longitude, anom.latitude, anom[v].where(oceano), levels=niveis_q,
                     cmap=cmocean.cm.balance, extend="both", transform=ccrs.PlateCarree())
    ax.set_title(f"Anomalia: {titulo} — {TEXTO_CASO}", fontsize=9)
axs.flat[-1].set_visible(False)
barra_cores(fig, cf, f"Anomalia de fluxo (W m$^{{-2}}$, positivo para baixo = para o oceano) vs. {TEXTO_CLIM}")
rodape(fig, "Fonte: ERA5 (Hersbach et al., 2020), via ARCO-ERA5. Fluxos acumulados horários convertidos para W m-2 (÷ 3600 s).")
salvar_figura(fig, f"t3_anomalia_fluxos_{ANO}.png")
plt.show()

## 10. Água doce e resposta oceânica

### 10.1 E − P (ERA5)

Média do caso e anomalia de **E − P** (evaporação menos precipitação), em mm/dia. E − P > 0: o oceano perde água doce, o que tende a **aumentar a salinidade**.

Conversão: o ERA5 dá `e` e `tp` em metros de água acumulados em 1 h. Fazemos E − P = (−e − tp) × 1000 × 24 mm/dia (o sinal de `e` é invertido porque a evaporação é negativa na convenção "positivo para baixo").

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5), subplot_kw=dict(projection=ccrs.PlateCarree()))
campos = [(media_caso["EmP"], f"E − P — média {TEXTO_CASO}"), (anom["EmP"], f"Anomalia de E − P vs. {TEXTO_CLIM}")]
for ax, (campo, titulo) in zip(axs, campos):
    mapa_base(ax, REGIAO, terra=True)
    lim = np.ceil(float(np.nanpercentile(np.abs(campo.where(oceano)), 99)))
    cf = ax.contourf(campo.longitude, campo.latitude, campo.where(oceano),
                     levels=np.linspace(-lim, lim, 21), cmap=cmocean.cm.balance, extend="both",
                     transform=ccrs.PlateCarree())
    fig.colorbar(cf, ax=ax, orientation="horizontal", pad=0.08, shrink=0.85, label="mm dia$^{-1}$")
    ax.set_title(titulo, fontsize=9)
rodape(fig, "Fonte: ERA5 (Hersbach et al., 2020), via ARCO-ERA5. E − P = −(e + tp), convertido de m h-1 para mm dia-1.")
salvar_figura(fig, f"t3_e_menos_p_{ANO}.png")
plt.show()

### 10.2 Anomalias do GLORYS12: TSM, SSS e camada de mistura

Anomalias da média do caso (médias mensais do GLORYS12) em relação à climatologia mensal do produto (1993–2016), nos mesmos meses. **O que observar:** a anomalia quente de TSM (onda de calor marinha) coincide com a anomalia positiva de Q? Onde a camada de mistura ficou **mais rasa** (anomalia negativa de `mlotst`) e onde ficou **mais funda**? Compare com os mapas de anomalia do vento e de E − P. A SSS acompanha o E − P?

**Cuidado ao interpretar `mlotst`:** no verão a camada de mistura tem só \~15 m, e entre 10 e 20 m os níveis do GLORYS12 estão separados por 2–3 m. Anomalias de 1–2 m equivalem a menos de um nível do modelo e são sensíveis ao critério usado para definir a camada (aqui, densidade).

In [ ]:
paineis = [("thetao", "TSM", "°C", 0.1), ("so", "SSS", "", 0.05), ("mlotst", "Camada de mistura", "m", 1.0)]
fig, axs = plt.subplots(1, 3, figsize=(17, 5), subplot_kw=dict(projection=ccrs.PlateCarree()))
for ax, (v, titulo, unid, arred) in zip(axs, paineis):
    mapa_base(ax, REGIAO, terra=True)
    lim = float(np.nanpercentile(np.abs(g_anom[v]), 99))
    lim = max(np.ceil(lim / arred) * arred, arred)
    cf = ax.contourf(g_anom.longitude, g_anom.latitude, g_anom[v], levels=np.linspace(-lim, lim, 21),
                     cmap=cmocean.cm.balance, extend="both", transform=ccrs.PlateCarree())
    cb = fig.colorbar(cf, ax=ax, orientation="horizontal", pad=0.08, shrink=0.85,
                      label=f"Anomalia de {titulo} {('(' + unid + ')') if unid else ''}")
    cb.ax.xaxis.set_major_locator(plt.MaxNLocator(5))
    ax.set_title(f"Anomalia de {titulo} — {TEXTO_CASO} vs. clim. 1993–2016", fontsize=9)
rodape(fig, "Fonte: GLORYS12 (Lellouche et al., 2021), CMEMS GLOBAL_MULTIYEAR_PHY_001_030 (médias mensais e climatologia mensal), via Copernicus Marine Toolbox (ARCO).")
salvar_figura(fig, f"t3_anomalias_glorys_{ANO}.png")
plt.show()

### 10.3 Opcional: série mensal numa caixa

Média na área da `CAIXA` (retângulo nos mapas) da TSM e da camada de mistura do GLORYS12, **mês a mês no ano do caso**, comparada com a climatologia mensal. Como o GLORYS12 é mensal, o custo é pequeno.

Fazer o mesmo para o fluxo líquido e a PNM do ERA5 exigiria ler o ano inteiro e mais 10 anos de climatologia (dezenas de milhares de campos globais), o que é lento demais para o Colab. Se quiser, adapte a seção 6 para outros meses.

In [ ]:
if FAZER_SERIE_CAIXA:
    g_ano = abre_glorys(ID_MENSAL, f"{ANO}-01-01", f"{ANO}-12-31", CAIXA)[["thetao", "mlotst"]].load()
    g_cl = copernicusmarine.open_dataset(
        dataset_id=ID_CLIM, variables=["thetao", "mlotst"],
        minimum_longitude=CAIXA["lon_o"], maximum_longitude=CAIXA["lon_l"],
        minimum_latitude=CAIXA["lat_s"], maximum_latitude=CAIXA["lat_n"],
        minimum_depth=0, maximum_depth=1,
        service="arco-geo-series", coordinates_selection_method="outside",
    ).isel(depth=0).load()

    # média na área, ponderada por cos(lat)
    def media_area(da):
        return da.weighted(np.cos(np.deg2rad(da.latitude))).mean(["latitude", "longitude"])

    fig, axs = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
    for ax, (v, rot) in zip(axs, [("thetao", "TSM (°C)"), ("mlotst", "Camada de mistura (m)")]):
        ax.plot(g_ano.time.dt.month, media_area(g_ano[v]), "o-", color="tab:red", label=str(ANO))
        ax.plot(g_cl.time.dt.month, media_area(g_cl[v]), "s--", color="k", label="climatologia 1993–2016")
        ax.axvspan(min(MESES) - 0.5, max(MESES) + 0.5, color="0.9", zorder=0)
        ax.set_ylabel(rot)
        ax.grid(lw=0.3)
        ax.legend(fontsize=8)
    axs[1].invert_yaxis()   # camada de mistura: mais funda para baixo
    axs[1].set_xticks(range(1, 13))
    axs[1].set_xticklabels([NOMES_MESES[m] for m in range(1, 13)])
    axs[0].set_title(f"Média na caixa {CAIXA['lat_s']}°–{CAIXA['lat_n']}°, {CAIXA['lon_o']}°–{CAIXA['lon_l']}°",
                     fontsize=9)
    fig.tight_layout()
    rodape(fig, "Fonte: GLORYS12 (Lellouche et al., 2021), CMEMS GLOBAL_MULTIYEAR_PHY_001_030.")
    salvar_figura(fig, f"t3_serie_caixa_{ANO}.png")
    plt.show()

### 10.4 Verificação de ordem de grandeza

In [ ]:
checagens = [
    ("PNM média (hPa)",           float(media_caso["msl"].min()),            float(media_caso["msl"].max()),            (960, 1040)),
    ("TSM média GLORYS12 (°C)",   float(g_mensal["thetao"].mean("time").min()), float(g_mensal["thetao"].mean("time").max()), (5, 31)),
    ("Onda curta média (W m-2)",  float(media_caso["ssr"].where(oceano).min()), float(media_caso["ssr"].where(oceano).max()), (0, 400)),
    ("Latente média (W m-2)",     float(media_caso["slhf"].where(oceano).min()), float(media_caso["slhf"].where(oceano).max()), (-300, 50)),
    ("Anomalia de Q (W m-2)",     float(anom["Q"].where(oceano).min()),      float(anom["Q"].where(oceano).max()),      (-150, 150)),
    ("E − P médio (mm/dia)",      float(media_caso["EmP"].where(oceano).min()), float(media_caso["EmP"].where(oceano).max()), (-30, 15)),
]
for nome, vmin, vmax, (esp_min, esp_max) in checagens:
    ok = esp_min <= vmin and vmax <= esp_max
    print(f"{nome:26s} {vmin:9.2f} a {vmax:8.2f}   esperado {esp_min} a {esp_max}   {'OK' if ok else 'VERIFICAR'}")

## 11. Para ir além

- **Climatologia do ERA5 mais robusta:** aumente `ANOS_CLIM` (com cuidado com o tempo de leitura) ou use as **médias mensais do ERA5** do Copernicus Climate Data Store, que são mais leves para climatologias longas (fora do escopo do acesso ARCO deste notebook).
- **TSM por satélite:** OSTIA ou NOAA OISST para confirmar a onda de calor marinha e calcular sua intensidade e duração (por exemplo, com o critério de Hobday et al., 2016).
- **Balanço de calor da camada de mistura:** compare a tendência de TSM com Q/(ρ c_p h), usando h = `mlotst`, para estimar quanto do aquecimento o fluxo local explica.
- **Altura geopotencial em 500 hPa** (`geopotential`, nível 500) para mostrar a crista/bloqueio em altitude. Atenção: no ARCO cada chunk tem os 37 níveis, então ler um nível custa o mesmo que ler todos.
- **Argo:** perfis de temperatura e salinidade na região para validar a camada de mistura do GLORYS12.

## Opcional: copiar dados e figuras para o Google Drive

A pasta `/content` some ao fim da sessão do Colab. Para guardar os resultados, mude `COPIAR_PARA_DRIVE` para `True` e rode a célula. O Colab vai pedir autorização para acessar o seu Drive.

In [ ]:
COPIAR_PARA_DRIVE = False   # >>> ALUNO: True para copiar
PASTA_NO_DRIVE = "ACA0430_trabalho1"   # >>> ALUNO: nome da pasta dentro de "Meu Drive"

if EM_COLAB and COPIAR_PARA_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    destino = Path("/content/drive/MyDrive") / PASTA_NO_DRIVE
    shutil.copytree(PASTA_DADOS, destino / "dados", dirs_exist_ok=True)
    shutil.copytree(PASTA_FIGURAS, destino / "figuras", dirs_exist_ok=True)
    print("Copiado para", destino)

## 12. Referências

**Dados e infraestrutura**
- Hersbach, H., et al. (2020). The ERA5 global reanalysis. *Quarterly Journal of the Royal Meteorological Society*, 146, 1999–2049. https://doi.org/10.1002/qj.3803
- Lellouche, J.-M., et al. (2021). The Copernicus Global 1/12° Oceanic and Sea Ice GLORYS12 Reanalysis. *Frontiers in Earth Science*, 9, 698876. https://doi.org/10.3389/feart.2021.698876
- Law-Chune, S., Aouf, L., Dalphinet, A., Levier, B., Drillet, Y., & Drevillon, M. (2021). WAVERYS: a CMEMS global wave reanalysis during the altimetry period. *Ocean Dynamics*, 71, 357–378. https://doi.org/10.1007/s10236-020-01433-w
- Abernathey, R. P., et al. (2021). Cloud-native repositories for big scientific data. *Computing in Science & Engineering*, 23(2), 26–35. https://doi.org/10.1109/MCSE.2021.3059437
- Documentação técnica (não revisada por pares, citar como fonte dos dados): repositório ARCO-ERA5 (https://github.com/google-research/arco-era5); Manual do Usuário CMEMS-GLO-PUM-001-030; página do produto GLOBAL_MULTIYEAR_WAV_001_032 (DOI 10.48670/moi-00022); documentação do Copernicus Marine Toolbox (https://toolbox-docs.marine.copernicus.eu).

**Tema**
- Coelho, C. A. S., de Oliveira, C. P., Ambrizzi, T., et al. (2016). The 2014 southeast Brazil austral summer drought: regional scale mechanisms and teleconnections. *Climate Dynamics*, 46, 3737–3752. https://doi.org/10.1007/s00382-015-2800-1
- Hobday, A. J., et al. (2016). A hierarchical approach to defining marine heatwaves. *Progress in Oceanography*, 141, 227–238. https://doi.org/10.1016/j.pocean.2015.12.014
- Reboita, M. S., Ambrizzi, T., Silva, B. A., Pinheiro, R. F., & da Rocha, R. P. (2019). The South Atlantic Subtropical Anticyclone: Present and future climate. *Frontiers in Earth Science*, 7, 8. https://doi.org/10.3389/feart.2019.00008
- Rodrigues, R. R., & Woollings, T. (2017). Impact of atmospheric blocking on South America in austral summer. *Journal of Climate*, 30(5), 1821–1837. https://doi.org/10.1175/JCLI-D-16-0493.1
- Rodrigues, R. R., Taschetto, A. S., Sen Gupta, A., & Foltz, G. R. (2019). Common cause for severe droughts in South America and marine heatwaves in the South Atlantic. *Nature Geoscience*, 12, 620–626. https://doi.org/10.1038/s41561-019-0393-8
- Silva, D. A., & Dottori, M. (2021). The atmospheric blocking influence over the South Brazil Bight during the 2013–2014 summer. *Regional Studies in Marine Science*, 45, 101815. https://doi.org/10.1016/j.rsma.2021.101815
- Sun, X., Cook, K. H., & Vizy, E. K. (2017). The South Atlantic Subtropical High: Climatology and interannual variability. *Journal of Climate*, 30, 3279–3296. https://doi.org/10.1175/JCLI-D-16-0705.1